L3–L4 sex comparisons: KS statistics, signed median shifts, original BH, combined ranks
and fixed highlighted-reaction comparisons. Existing samples and recorded reaction
exclusions are retained.

Set the relative input/output constants in the first code cell. No local Python files
are imported.


In [ ]:
from scipy.stats import false_discovery_control
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, spearmanr
from IPython.display import display

INPUT_DIR = Path("inputs")
RESULT_DIR = Path("outputs")
OUTPUT_DIR = RESULT_DIR / "comparison_results"
REFERENCE_PATH = INPUT_DIR / "haeconGEM1.xml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED = [
    "MAR00023",
    "MAR01506",
    "MAR01507",
    "MAR02960",
    "MAR02993",
    "MAR03031",
    "MAR03036",
    "MAR03041",
    "MAR03131",
    "MAR06630",
    "MAR06631",
    "MAR06632",
    "MAR06633",
    "MAR06634",
    "MAR06635",
    "MAR06636",
    "MAR06637",
    "MAR06638",
    "MAR06639",
    "MAR06641",
    "MAR06642",
    "MAR06646",
    "MAR06647",
    "MAR06648",
    "MAR06649",
    "MAR06651",
    "MAR06652",
    "MAR06653",
    "MAR06654",
    "MAR06655",
    "MAR06656",
    "MAR06657",
    "MAR06658",
    "MAR06659",
    "MAR06660",
    "MAR06662",
    "MAR06663",
    "MAR06664",
    "MAR06665",
    "MAR06666",
    "MAR06667",
    "MAR06668",
    "MAR06669",
    "MAR06670",
    "MAR06671",
    "MAR06672",
    "MAR06673",
    "MAR06674",
    "MAR06675",
    "MAR06676",
    "MAR06677",
    "MAR06678",
    "MAR06680",
    "MAR06682",
    "MAR06685",
    "MAR06686",
    "MAR06687",
    "MAR06690",
    "MAR06691",
    "MAR06692",
    "MAR06693",
    "MAR06694",
    "MAR06695",
    "MAR06697",
    "MAR06699",
    "MAR06700",
    "MAR06701",
    "MAR06702",
    "MAR06703",
    "MAR06704",
    "MAR06705",
    "MAR08697",
    "MAR08698",
    "MAR08699",
    "MAR08702",
    "MAR08703",
    "MAR08704",
    "MAR08706",
    "MAR08709",
    "MAR08710",
    "MAR08711",
    "MAR08712",
    "MAR08716",
    "MAR11894",
    "MAR11895",
    "MAR11896",
    "MAR11897",
    "MAR20001",
    "MAR20003",
    "MAR20005",
    "MAR20006",
    "MAR20007",
    "MAR20008",
]


def drop_cols(frame):
    """Remove recorded stage/sex IDs from a reactions-as-columns frame."""
    return frame.drop(columns=[rid for rid in EXCLUDED if rid in frame.columns])


CHRR = INPUT_DIR / "CHRR_sampling"
PCA = CHRR / "pca"
UNION = CHRR / "union_L4fm"
RANGE_MIN = 0.001
SIGN_TOL = 1e-09
SAMPLE_PATHS = {
    "L3": PCA / "L3_c_samples.csv",
    "L4f": PCA / "L4f_c_samples.csv",
    "L4m": PCA / "L4m_c_samples.csv",
    "L4union": UNION / "L4mf_union_c_samples.csv",
}
COMPARATORS = ["L4union", "L4f", "L4m"]
HIGHLIGHTS = ["MAR04579", "MAR04855", "MAR09067", "MAR03899", "MAR09064"]
SHORT_NAMES = {
    "MAR04579": "Uridine phosphorylase",
    "MAR04855": "Fumarate/malate antiporter",
    "MAR09067": "Glycine exchange",
    "MAR03899": "Alanine aminotransferase",
    "MAR09064": "Tyrosine exchange",
}


def save(data, name):
    d = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    d.to_csv(OUTPUT_DIR / name, sep="\t", index=False)
    return d


samples = {}
sample_inventory = []
for stage, path in SAMPLE_PATHS.items():
    raw = pd.read_csv(path)
    samples[stage] = drop_cols(raw)
    sample_inventory.append(
        dict(
            model=stage,
            samples=len(raw),
            sample_reactions_before_exclusion=raw.shape[1],
            sample_reactions_after_exclusion=samples[stage].shape[1],
            excluded_reaction_ids=";".join(
                sorted(set(raw.columns) - set(samples[stage].columns))
            ),
        )
    )
ev = (
    pd.read_csv(PCA / "reaction_evidence_table.csv")
    .drop_duplicates("rxnID")
    .set_index("rxnID")
)
ranges = {k: d.max() - d.min() for k, d in samples.items()}
save(sample_inventory, "sample_inventory.tsv")




def shift_sign(x):
    return (
        "comparator_higher"
        if x > SIGN_TOL
        else "L3_higher" if x < -SIGN_TOL else "numerically_tied"
    )


def annotate(df):
    for column in ["probable_name", "subSystem", "equation"]:
        df[column] = df.rxnID.map(ev[column])
    return df


In [ ]:
results = {}
eligibility = []
comparison_rows = []
a = samples["L3"]
for other in COMPARATORS:
    b = samples[other]
    shared = sorted(set(a.columns) & set(b.columns))
    eligible = []
    for rid in shared:
        ok = ranges["L3"][rid] >= RANGE_MIN and ranges[other][rid] >= RANGE_MIN
        eligibility.append(
            dict(
                comparison="L3_vs_" + other,
                reaction_id=rid,
                range_L3=ranges["L3"][rid],
                range_comparator=ranges[other][rid],
                eligible=ok,
            )
        )
        if ok:
            eligible.append(rid)
    rows = []
    for rid in eligible:
        D, p = ks_2samp(a[rid].values, b[rid].values)
        m1 = a[rid].median()
        m2 = b[rid].median()
        span = max(a[rid].max(), b[rid].max()) - min(a[rid].min(), b[rid].min())
        rows.append((rid, D, p, m1, m2, m2 - m1, span))
    d = pd.DataFrame(
        rows,
        columns=[
            "rxnID",
            "D",
            "pvalue",
            "median_L3",
            "median_comparator",
            "delta_median",
            "pooled_range",
        ],
    )
    d["qvalue"] = false_discovery_control(d.pvalue)
    d["relative_delta"] = d.delta_median.abs() / d.pooled_range
    d["rank_D"] = d.D.rank(ascending=False, method="min").astype(int)
    d["rank_relative_delta"] = d.relative_delta.rank(
        ascending=False, method="min"
    ).astype(int)
    d["combined_score"] = d.rank_D + d.rank_relative_delta
    d = d.sort_values("combined_score").reset_index(drop=True)
    d["median_shift_direction"] = d.delta_median.map(shift_sign)
    d["comparison"] = "L3_vs_" + other
    results[other] = annotate(d)
    save(results[other], f"pairwise_ks_L3_vs_{other}.tsv")
    comparison_rows.append(
        dict(
            comparison="L3_vs_" + other,
            samples_L3=len(a),
            samples_comparator=len(b),
            shared_after_original_exclusions=len(shared),
            tested_reactions=len(d),
            excluded_by_original_range_filter=len(shared) - len(d),
            D_equal_1=int(d.D.eq(1).sum()),
            median_D=d.D.median(),
            median_relative_delta=d.relative_delta.median(),
            comparator_higher=int(
                d.median_shift_direction.eq("comparator_higher").sum()
            ),
            L3_higher=int(d.median_shift_direction.eq("L3_higher").sum()),
            numerically_tied=int(d.median_shift_direction.eq("numerically_tied").sum()),
            q_below_0_05_computational_only=int(d.qvalue.lt(0.05).sum()),
        )
    )
save(eligibility, "all_shared_reaction_eligibility.tsv")
comparison_summary = save(comparison_rows, "comparison_summary.tsv")
display(comparison_summary)


In [ ]:
highlights = []
joined = []
for other in COMPARATORS:
    d = results[other].set_index("rxnID")
    for rid in HIGHLIGHTS:
        row = dict(reaction_id=rid, name=SHORT_NAMES[rid], comparison="L3_vs_" + other)
        present_L3 = rid in samples["L3"]
        present_other = rid in samples[other]
        row.update(present_in_L3=present_L3, present_in_comparator=present_other)
        if rid in d.index:
            r = d.loc[rid]
            row.update(
                status="eligible",
                D=r.D,
                median_L3=r.median_L3,
                median_comparator=r.median_comparator,
                delta_median=r.delta_median,
                relative_delta=r.relative_delta,
                rank_D=r.rank_D,
                rank_relative_delta=r.rank_relative_delta,
                combined_score=r.combined_score,
                median_shift_direction=r.median_shift_direction,
            )
            reference = results["L4union"].set_index("rxnID").loc[rid]
            row["same_direction_as_union"] = (
                r.median_shift_direction == reference.median_shift_direction
            )
        else:
            row["status"] = (
                "absent"
                if not (present_L3 and present_other)
                else "range_below_original_filter"
            )
        highlights.append(row)
    x = d[
        [
            "D",
            "median_L3",
            "median_comparator",
            "delta_median",
            "relative_delta",
            "rank_D",
            "rank_relative_delta",
            "combined_score",
            "median_shift_direction",
        ]
    ].copy()
    x.columns = [other + "_" + k for k in x.columns]
    joined.append(x)
wide = pd.concat(joined, axis=1, join="outer").rename_axis("reaction_id").reset_index()
save(wide, "all_tested_reactions_joined.tsv")
common_ids = sorted(set.intersection(*(set(results[k].rxnID) for k in COMPARATORS)))
common = wide.set_index("reaction_id").loc[common_ids].reset_index()
agreements = []
for other in ["L4f", "L4m"]:
    common[other + "_same_direction_as_union"] = (
        common[other + "_median_shift_direction"]
        == common.L4union_median_shift_direction
    )
    ref = set(
        common.loc[
            common.L4union_median_shift_direction == "comparator_higher", "reaction_id"
        ]
    )
    sex = set(
        common.loc[
            common[other + "_median_shift_direction"] == "comparator_higher",
            "reaction_id",
        ]
    )
    agreements.append(
        dict(
            comparison="L3_vs_" + other,
            common_eligible_reactions=len(common),
            same_direction=int(common[other + "_same_direction_as_union"].sum()),
            opposite_direction=int(
                (
                    (common.L4union_delta_median * common[other + "_delta_median"] < 0)
                    & (common.L4union_delta_median.abs() > SIGN_TOL)
                    & (common[other + "_delta_median"].abs() > SIGN_TOL)
                ).sum()
            ),
            direction_agreement_fraction=common[
                other + "_same_direction_as_union"
            ].mean(),
            spearman_D=float(
                spearmanr(common.L4union_D, common[other + "_D"]).statistic
            ),
            spearman_relative_delta=float(
                spearmanr(
                    common.L4union_relative_delta, common[other + "_relative_delta"]
                ).statistic
            ),
            comparator_higher_set_intersection=len(ref & sex),
            comparator_higher_set_gained_vs_union=len(sex - ref),
            comparator_higher_set_lost_vs_union=len(ref - sex),
            comparator_higher_set_jaccard=(
                len(ref & sex) / len(ref | sex) if ref | sex else 1.0
            ),
            direction_disagreement_ids=";".join(
                sorted(
                    common.loc[
                        ~common[other + "_same_direction_as_union"], "reaction_id"
                    ]
                )
            ),
        )
    )
highlights = save(highlights, "paper_highlighted_reaction_comparisons.tsv")
save(common, "common_eligible_reaction_comparison.tsv")
agreement = save(agreements, "direction_and_effect_agreement_vs_union.tsv")
highlight_table = []
for rid in HIGHLIGHTS:
    row = dict(reaction_id=rid, reaction=SHORT_NAMES[rid])
    for other in COMPARATORS:
        r = highlights[
            (highlights.reaction_id == rid)
            & (highlights.comparison == "L3_vs_" + other)
        ].iloc[0]
        row[other + "_D"] = r.get("D", np.nan)
        row[other + "_delta_median"] = r.get("delta_median", np.nan)
        row[other + "_direction"] = r.get("median_shift_direction", r.status)
    highlight_table.append(row)
highlight_table = save(highlight_table, "paper_highlighted_reactions_side_by_side.tsv")
display(agreement)
display(highlight_table)
